# 00 - Data Overview

What the fine-tuning stage learns from, before any model is mentioned. It reads `data/processed/odor/odor_dataset.csv`
and the summary `output/analysis/eda.json`, so it needs no GPU and no training.

Run it from the repository root, or from `notebooks/`, and it finds the paths itself.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
table = pd.read_csv(ROOT / "data/processed/odor/odor_dataset.csv").fillna("")
eda = json.loads((ROOT / "output/analysis/eda.json").read_text())

tokens = [t for text in table["odor_tokens"] for t in str(text).split(";") if t]
per_molecule = [len([t for t in str(text).split(";") if t]) for text in table["odor_tokens"]]
print(f"{len(table):,} molecules, {len(set(tokens))} odor tokens, {sum(per_molecule) / len(table):.1f} tokens a molecule")

## 1. Split sizes

The fine-tuning set is split once, by CID, with a fixed seed. The DREAM molecules are held out of everything,
including the pretraining.

In [ ]:
counts = table["split"].value_counts().reindex(["train", "validation", "test", "dream"])
fig, ax = plt.subplots(figsize=(6, 3))
counts.plot.bar(ax=ax, color=["#1F3864", "#4472C4", "#4472C4", "#8FAADC"])
ax.set_ylabel("molecules")
ax.set_title("7,516 molecules with odor tokens")
for i, value in enumerate(counts):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. The vocabulary is long-tailed

890 odor tokens, but the molecules are not spread evenly over them. A few terms carry thousands of molecules and
211 terms sit on exactly one.

In [ ]:
usage = pd.Series(tokens).value_counts()
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(usage.values, bins=np.logspace(0, np.log10(usage.max()), 30), color="#1F3864")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("molecules carrying the token")
ax.set_ylabel("tokens")
ax.set_title(f"{len(usage)} odor tokens, {(usage == 1).sum()} on a single molecule")
plt.tight_layout()
plt.show()

print(usage.head(8).to_string())

### What the long tail costs

The histogram says there is a tail; this says how much of the data the head carries. Tokens sorted by how many
molecules carry them, cumulatively.

In [ ]:
ordered = np.sort(usage.values)[::-1]
covered = np.cumsum(ordered) / ordered.sum()
most = int(np.searchsorted(covered, 0.9)) + 1

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(covered) + 1), covered, color="#1F3864")
ax.axhline(0.9, color="#C00000", linestyle="--", linewidth=1)
ax.axvline(most, color="#C00000", linestyle="--", linewidth=1, label=f"{most} tokens carry 90%")
ax.set_xlabel("odor tokens, most used first")
ax.set_ylabel("share of the label mass")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"{most} of {len(covered)} tokens carry 90 percent of the labels")

## 3. How many odor tokens a molecule carries

Median four, mean 5.8, and one molecule with 34. The generator is asked for one to six of them at a time.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(per_molecule, bins=range(0, max(per_molecule) + 2), color="#4472C4", align="left")
ax.axvline(np.median(per_molecule), color="#C00000", linestyle="--", linewidth=1, label=f"median {np.median(per_molecule):.0f}")
ax.set_xlabel("odor tokens on a molecule")
ax.set_ylabel("molecules")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 4. The sources

Nine, not eleven. `flavordb` is commented out in `SOURCES`, and Dravnieks and Keller never supplied odor tokens;
the DREAM list is only the held-out set.

In [ ]:
sources = pd.Series(eda["sources"]).sort_values()
fig, ax = plt.subplots(figsize=(6, 3.2))
sources.plot.barh(ax=ax, color="#1F3864")
ax.set_xlabel("molecules")
ax.set_title("molecules named by each source")
plt.tight_layout()
plt.show()

## 5. What the molecules weigh

The pretraining keeps molecules of 350 Da or less, because a molecule has to evaporate to be smelled. The
fine-tuning set is not filtered that way: it is what the sources name.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

weights = [Descriptors.MolWt(Chem.MolFromSmiles(s)) for s in table["smiles"] if Chem.MolFromSmiles(s)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(weights, bins=60, color="#4472C4")
ax.axvline(350, color="#C00000", linestyle="--", linewidth=1, label="350 Da")
ax.set_xlabel("molecular weight (Da)")
ax.set_ylabel("molecules")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print(f"median {np.median(weights):.0f} Da, {np.mean(np.array(weights) > 350):.1%} over 350")

## Reading.

Seven and a half thousand molecules, and the two halves of the problem are visible in the counts. The token
vocabulary is wide and thin: a few terms such as `fruity` and `sweet` are attached to thousands of molecules, and
211 terms to one, which bounds what any classifier can do with them. 193 of the 890 tokens carry 90 percent of the label mass;
the other 697 share the rest. Each molecule carries about six tokens, and
the generator is trained on requests of one to six, so it never sees a molecule's full description at once.

The weights are the other half. 98.5 percent of the molecules sit under 350 Da, which is why the pretraining
keeps only small ones; the 1.5 percent above it are kept in the fine-tuning set because the sources name them.

## Verdict

The data is small, unbalanced and long-tailed, and the splits are clean. Anything the models score above chance on
the single-molecule tokens is luck or leakage, and the report should say so before showing a macro average.
